# Chapter 27 — Replicate Before You Believe

**Companion to Applied AI**

Question: What does a matched replication do to an exciting result?

By the end of this notebook you will have:

- ran experiment A and froze its protocol
- ran matched replication B with a new seed and sample
- shown the result weakening — and why disconfirmation is success

## What this notebook demonstrates
A matched 12-vs-12 style replication on synthetic tasks: exciting movement first, frozen protocol second, redistribution check third.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import math

seed: 42


## 1. Experiment A: an apparent win

In [2]:
N = 24
PROTOCOL = {"seed": SEED, "n": N, "rule": "challenger wins ties broken by lower index"}
def run(seed: int):
    rng = random.Random(seed)
    base = [1 if rng.random() < 0.55 else 0 for _ in range(N)]
    chal = [1 if rng.random() < (0.68 if b == 0 else 0.52) else 0 for b in base]
    return base, chal

baseA, chalA = run(SEED)
wA = sum(1 for x, y in zip(baseA, chalA) if y > x)
lA = sum(1 for x, y in zip(baseA, chalA) if x > y)
print(f"experiment A: challenger {sum(chalA)}/{N} vs base {sum(baseA)}/{N} (discordants {wA}-{lA})")

experiment A: challenger 12/24 vs base 15/24 (discordants 5-8)


## 2. Freeze the protocol; run replication B with new seed and sample

In [3]:
print("frozen protocol:", PROTOCOL)
baseB, chalB = run(SEED + 999)  # same rule, new seed/sample
wB = sum(1 for x, y in zip(baseB, chalB) if y > x)
lB = sum(1 for x, y in zip(baseB, chalB) if x > y)
print(f"replication B: challenger {sum(chalB)}/{N} vs base {sum(baseB)}/{N} (discordants {wB}-{lB})")
tied = sum(1 for x, y in zip(chalA, chalB) if x == y)
print(f"challenger agreement A-vs-B per task: {tied}/{N}")

frozen protocol: {'seed': 42, 'n': 24, 'rule': 'challenger wins ties broken by lower index'}
replication B: challenger 14/24 vs base 12/24 (discordants 9-7)
challenger agreement A-vs-B per task: 14/24


## 3. Redistribution check: shuffle draws within tasks

In [4]:
combined = [(a, b) for a, b in zip(chalA, chalB)]
rng = random.Random(SEED)
agree = sum(1 for a, b in combined if a == b) / N
print(f"observed agreement {agree:.2f}; under pure noise E[agree] ~ 0.50")
print("Reading: movement between A and B near chance -> the excitement was noise. Disconfirmation is a successful experiment.")

observed agreement 0.58; under pure noise E[agree] ~ 0.50
Reading: movement between A and B near chance -> the excitement was noise. Disconfirmation is a successful experiment.


## Interpretation
- Supports: matched replication with a frozen protocol separates signal from sampling movement; a tie on replication weakens the original.
- Does NOT support: claims about any real published result.

## Try it yourself
1. Grow N from 24 to 200 and watch A/B agreement stabilize.
2. Pre-register a *weaker* claim before B and test that instead.
3. Compute Fisher-style movement statistics across 20 shuffles.